# ДЗ 7.1. Остывание стержня — можно ли доверять устойчивой схеме?

Решаем $u_t=u_{xx}$ на $x\in[0,1]$ при $u(0,t)=u(1,t)=0$.

- FTCS и неявный Эйлер: сравнение с точным решением, сетки $N=25,50,100$, $r=0.4$, моменты $t=0.001,0.01,0.1$; ошибка $E_\infty$, порядок сходимости.
- Кранк—Николсон; начальный нагретый участок $[0.4,0.6]$, $N=100$, $r=0.4,2,10$; профили первых слоёв, увеличенные фрагменты, min/max, сравнение при одинаковых физических временах.

Шаг по времени:$\Delta t=r\Delta x^2$

## 1. Схемы и критерии

Для внутренних узлов $i=1,\dots,N-1$, $\Delta x=1/N$, $r=\Delta t/\Delta x^2$:

$$
\begin{aligned}
\text{FTCS:}\quad &U_i^{n+1}=U_i^n+r(U_{i-1}^n-2U_i^n+U_{i+1}^n),\\
\text{Эйлер назад:}\quad &(I-rL)U^{n+1}=U^n,\\
\text{Кранк--Николсон:}\quad &(I-\tfrac r2 L)U^{n+1}=(I+\tfrac r2L)U^n,
\end{aligned}
$$

где $(LU)_i=U_{i-1}-2U_i+U_{i+1}$ (без деления на $\Delta x^2$).

**Устойчивость:** FTCS требует $r\le 1/2$; обе неявные схемы устойчивы при любом $r>0$ в $\ell_2$. **Но** устойчивость не гарантирует точность и отсутствие немонотонных колебаний: Кранк—Николсон не является безусловно монотонной схемой.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.linalg import solve_banded
from functools import lru_cache
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 115,
    'font.size': 10,
    'axes.grid': True,
    'grid.alpha': 0.22,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

@lru_cache(maxsize=None)
def banded_matrix(method, size, r):
    """Трёхдиагональная левая часть в формате solve_banded."""
    ab = np.zeros((3, size), dtype=float)
    if method == 'BE':
        a, b = -r, 1 + 2*r
    elif method == 'CN':
        a, b = -r/2, 1 + r
    else:
        raise ValueError('Для матрицы доступны BE и CN')
    ab[0, 1:] = a
    ab[1, :] = b
    ab[2, :-1] = a
    return ab


def step(u, method, r):
    """Один шаг. Граничные значения x=0,1 всегда равны нулю."""
    v = np.zeros_like(u)
    center = u[1:-1]
    lap = u[:-2] - 2*center + u[2:]
    if method == 'FTCS':
        v[1:-1] = center + r*lap
    elif method in ('BE', 'CN'):
        rhs = center if method == 'BE' else center + 0.5*r*lap
        v[1:-1] = solve_banded(
            (1, 1), banded_matrix(method, center.size, float(r)),
            rhs, check_finite=False,
        )
    else:
        raise ValueError('method должен быть FTCS, BE или CN')
    return v


def solve_to_time(u0, method, r, t):
    """Шаг dt = r*dx². Остаток до t проводится укороченным шагом."""
    dx = 1/(len(u0)-1)
    dt = r*dx**2
    q = t/dt
    n_full = int(round(q)) if abs(q-round(q)) < 1e-10 else int(np.floor(q))
    u = np.array(u0, dtype=float, copy=True)
    for _ in range(n_full):
        u = step(u, method, r)
    dt_last = t - n_full*dt
    if dt_last > 1e-13:
        u = step(u, method, dt_last/dx**2)
    return u


def trajectory(u0, method, r, t_end):
    """Все полные временные слои, когда t_end кратно dt."""
    dx = 1/(len(u0)-1)
    dt = r*dx**2
    n_steps = int(round(t_end/dt))
    assert np.isclose(n_steps*dt, t_end, atol=1e-12, rtol=0), 'Не кратное dt время'
    values = np.empty((n_steps+1, len(u0)))
    values[0] = u0
    for k in range(1, n_steps+1):
        values[k] = step(values[k-1], method, r)
    times = np.arange(n_steps+1)*dt
    return times, values


def at_time(record, t):
    """Выбор слоя с точно совпадающим физическим временем."""
    times, values = record
    j = int(np.argmin(np.abs(times-t)))
    assert np.isclose(times[j], t, atol=1e-11, rtol=0)
    return values[j]

## 2. Проверка по точному решению

$$u(x,0)=\sin(\pi x)+0.3\sin(5\pi x),\qquad
u(x,t)=e^{-\pi^2t}\sin(\pi x)+0.3e^{-25\pi^2t}\sin(5\pi x).$$

$E_\infty=\max_i|U_i-u(x_i,t)|$; при уменьшении $\Delta x$ вдвое одновременно уменьшается $\Delta t$ **вчетверо** (так как $r=0.4$ фиксировано). Оценка порядка $p=\log_2(E_N/E_{2N})$.

In [ ]:
def smooth_initial(x):
    return np.sin(np.pi*x) + 0.3*np.sin(5*np.pi*x)

def exact(x, t):
    return (np.exp(-np.pi**2*t)*np.sin(np.pi*x)
            + 0.3*np.exp(-25*np.pi**2*t)*np.sin(5*np.pi*x))

Ns = (25, 50, 100)
targets = (0.001, 0.01, 0.1)
methods_green = ('FTCS', 'BE')
r_green = 0.4

errors = {}
solutions = {}
for method in methods_green:
    for t in targets:
        for N in Ns:
            x = np.linspace(0, 1, N+1)
            u_num = solve_to_time(smooth_initial(x), method, r_green, t)
            solutions[method, t, N] = u_num
            errors[method, t, N] = np.max(np.abs(u_num - exact(x, t)))

rows = []
for method in methods_green:
    for t in targets:
        e25, e50, e100 = [errors[method, t, N] for N in Ns]
        rows.append({
            'Схема': 'FTCS' if method == 'FTCS' else 'Эйлер назад',
            't': t,
            'E∞, N=25': f'{e25:.3e}',
            'E∞, N=50': f'{e50:.3e}',
            'E∞, N=100': f'{e100:.3e}',
            'p(25→50)': round(np.log2(e25/e50), 3),
            'p(50→100)': round(np.log2(e50/e100), 3),
        })
errors_table = pd.DataFrame(rows)
display(errors_table.style.hide(axis='index').set_caption('Максимальные ошибки и экспериментальные порядки'))

assert all(errors[m, t, 25] > errors[m, t, 50] > errors[m, t, 100]
           for m in methods_green for t in targets)
print('✓ При всех трёх временах и обоих методах ошибка убывает при сгущении сетки.')

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(13, 10), sharex=True)
for i, t in enumerate(targets):
    for j, method in enumerate(methods_green):
        ax = axs[i, j]
        fine_x = np.linspace(0, 1, 1001)
        ax.plot(fine_x, exact(fine_x, t), color='black', lw=2.3, label='Точное')
        for N, marker in [(25, 'o'), (50, 's'), (100, '.')]:
            xx = np.linspace(0, 1, N+1)
            ax.plot(xx, solutions[method, t, N], marker=marker, ms=2.8,
                    lw=1, markevery=max(1, N//15), label=f'N={N}')
        ax.set(title=f'{"FTCS" if method == "FTCS" else "Эйлер назад"},  t={t:g}',
               ylabel='u(x,t)', xlim=(0,1))
        if i == 0:
            ax.legend(ncol=2, fontsize=8)
for ax in axs[-1]:
    ax.set_xlabel('x')
fig.suptitle('Численные и точные профили в одинаковые моменты времени', fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12.5, 4.2), sharey=True)
for ax, method in zip(axs, methods_green):
    for t in targets:
        ax.loglog(np.array(Ns, float), [errors[method, t, N] for N in Ns],
                  '-o', label=f't={t:g}')
    ax.set(title=method, xlabel='Число интервалов N', ylabel='$E_\infty$')
    ax.set_xticks(Ns, labels=[str(n) for n in Ns])
    ax.legend()
fig.suptitle('Сходимость при фиксированном r=0.4:  Δt ∝ Δx²', fontsize=12)
fig.tight_layout()
plt.show()

**Интерпретация.** Обе схемы имеют пространственную погрешность $O(\Delta x^2)$ и временную $O(\Delta t)$. При $\Delta t=0.4\Delta x^2$ общая погрешность — $O(\Delta x^2)$, поэтому наблюдаемый $p\to2$. Для самых грубых сеток и $t=0.001$ порядок может отличаться от 2 из-за ещё заметной пятой гармоники и укороченного последнего шага. Пятая гармоника затухает как $e^{-25\pi^2t}$, первая — как $e^{-\pi^2t}$.

### Анимация 1 — точный профиль, FTCS и неявный Эйлер

$N=100$, $r=0.4$. Все кадры соответствуют одинаковым $t\in[0,0.1]$ с шагом $0.002$.

In [ ]:
N_anim = 100
xx = np.linspace(0, 1, N_anim+1)
u_init = smooth_initial(xx)
rec_ftcs = trajectory(u_init, 'FTCS', 0.4, 0.1)
rec_be = trajectory(u_init, 'BE', 0.4, 0.1)
frame_times = np.linspace(0, 0.1, 51)

fig, ax = plt.subplots(figsize=(9.5, 4.7))
line_true, = ax.plot([], [], 'k-', lw=2.4, label='Точное')
line_ftcs, = ax.plot([], [], '-', lw=1.8, label='FTCS')
line_be, = ax.plot([], [], '--', lw=1.8, label='Эйлер назад')
ax.set(xlim=(0,1), ylim=(-0.04, 1.38), xlabel='x', ylabel='u(x,t)')
ax.legend(loc='upper right')
clock = ax.text(0.025, 0.92, '', transform=ax.transAxes, fontsize=11)

def update_green(frame):
    t = float(frame_times[frame])
    line_true.set_data(xx, exact(xx, t))
    line_ftcs.set_data(xx, at_time(rec_ftcs, t))
    line_be.set_data(xx, at_time(rec_be, t))
    clock.set_text(f't = {t:.3f}')
    return line_true, line_ftcs, line_be, clock

anim_green = FuncAnimation(fig, update_green, frames=len(frame_times),
                           interval=100, blit=True, repeat=True)
display(HTML(anim_green.to_jshtml(fps=10, default_mode='loop')))
plt.close(fig)

## 3. Разрывный профиль: нагретый участок

$$u(x,0)=\begin{cases}1,&0.4\le x\le 0.6,\\0,&\text{иначе.}\end{cases}$$

**Одна и та же сетка** $N=100$, $\Delta x=0.01$. Исследуем неявный Эйлер и Кранк—Николсон при $r\in\{0.4,2,10\}$. Для обоих методов допускаются большие шаги; явный FTCS при $r=2,10$ не используем, так как он неустойчив. Отрицательные значения **не отсекаются**.

In [ ]:
N_heat = 100
x_heat = np.linspace(0, 1, N_heat+1)
heat0 = np.where((x_heat >= 0.4-1e-12) & (x_heat <= 0.6+1e-12), 1.0, 0.0)
rs = (0.4, 2.0, 10.0)
methods_heat = ('BE', 'CN')
heat_records = {}
for method in methods_heat:
    for r in rs:
        heat_records[method, r] = trajectory(heat0, method, r, 0.1)

dt_table = pd.DataFrame({
    'r': rs,
    'Δx': [1/N_heat]*3,
    'Δt = r Δx²': [r/N_heat**2 for r in rs],
    'Шагов до t=0.1': [int(round(0.1*N_heat**2/r)) for r in rs],
})
display(dt_table.style.hide(axis='index').format({'Δx':'{:.2f}', 'Δt = r Δx²':'{:.5f}'}))

### Первые пять временных слоёв

В **каждом** из шести графиков изображены слои $n=0,\dots,4$. Внутри отдельного графика шаг одинаков, но **между столбцами** времена слоёв различаются, поскольку $\Delta t=r\Delta x^2$. Поэтому графики ниже служат для диагностики начального поведения, а не для сравнения решений в один и тот же момент.

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(14.5, 7.8), sharex=True, sharey=True)
for i, method in enumerate(methods_heat):
    for j, r in enumerate(rs):
        ax = axs[i, j]
        times, vals = heat_records[method, r]
        for k in range(5):
            ax.plot(x_heat, vals[k], lw=1.5, label=f'n={k}, t={times[k]:.5f}')
        ax.set(title=f'{"Эйлер назад" if method == "BE" else "Кранк—Николсон"}, r={r:g}',
               xlim=(0.25, 0.75), ylim=(-0.09, 1.09))
        ax.legend(fontsize=7, loc='upper right')
        if j == 0:
            ax.set_ylabel('u(x,t)')
for ax in axs[-1]:
    ax.set_xlabel('x')
fig.suptitle('Нагретый участок: первые слои (включая начальный)', fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(14.5, 7.1), sharey=True)
for col, r in enumerate(rs):
    times, vals = heat_records['CN', r]
    for row, bounds in enumerate(((0.36, 0.44), (0.56, 0.64))):
        ax = axs[row, col]
        for k in range(5):
            ax.plot(x_heat, vals[k], marker='.', ms=3, lw=1.3, label=f'n={k}')
        ax.axvline(0.4 if row==0 else 0.6, ls=':', color='black', lw=1)
        ax.set(xlim=bounds, ylim=(-0.06,1.06),
               title=f'Кранк—Николсон, r={r:g}; {"левая" if row == 0 else "правая"} граница',
               xlabel='x')
        if col == 0:
            ax.set_ylabel('u(x,t)')
        if row == 0:
            ax.legend(ncol=2, fontsize=8)
fig.suptitle('Увеличение у x=0.4 и x=0.6: заметно чередование слоёв при больших r', fontsize=12)
fig.tight_layout()
plt.show()

### Min/max во времени и колебания на границе участка

Минимум и максимум вычисляются **по всей сетке**, включая граничные узлы. Из-за $u(0,t)=u(1,t)=0$ минимум часто равен нулю и может не выявлять локальную немонотонность: поэтому отдельно отслеживаем узел $x=0.4$.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4.4))
i_edge = int(round(0.4*N_heat))
for method in methods_heat:
    for r in rs:
        times, vals = heat_records[method, r]
        label = f'{"Эйлер" if method == "BE" else "КН"}, r={r:g}'
        style = '-' if method == 'BE' else '--'
        axs[0].plot(times, vals.min(axis=1), ls=style, label=label)
        axs[1].plot(times, vals.max(axis=1), ls=style, label=label)
        axs[2].plot(times, vals[:,i_edge], ls=style, label=label)
axs[0].set(title='min по x', xlabel='t', ylabel='min u', ylim=(-0.05,0.1))
axs[1].set(title='max по x', xlabel='t', ylabel='max u')
axs[2].set(title='Узел x=0.4', xlabel='t', ylabel='u(0.4,t)', xlim=(0,0.025))
axs[1].legend(fontsize=8, ncol=2)
fig.suptitle('Проверка ограниченности и временной монотонности', fontsize=13)
fig.tight_layout()
plt.show()

### Сопоставление при одинаковых моментах времени

За **эталон для сравнения, а не точное решение**, взят неявный Эйлер с $r=0.4$ на той же сетке. Во всех строках $t=0.002,0.01,0.1$ — строго общие физические моменты, кратные всем трём шагам. Разность $D_\infty=\max_i|U_i^{\mathrm{method},r}(t)-U_i^{\mathrm{BE},0.4}(t)|$.

In [ ]:
comparison_times = (0.002, 0.01, 0.1)
compare_rows = []
for t in comparison_times:
    reference = at_time(heat_records['BE',0.4], t)
    for method in methods_heat:
        for r in rs:
            u = at_time(heat_records[method,r], t)
            compare_rows.append({
                't': t,
                'Метод': 'Эйлер назад' if method=='BE' else 'Кранк—Николсон',
                'r': r,
                'min u': float(np.min(u)),
                'max u': float(np.max(u)),
                'u(0.4,t)': float(u[i_edge]),
                'D∞ к BE (r=0.4)': float(np.max(np.abs(u-reference))),
            })
comparison = pd.DataFrame(compare_rows)
display(comparison.style.hide(axis='index').format({
    't':'{:.3f}', 'r':'{:.1f}', 'min u':'{:.4f}', 'max u':'{:.4f}',
    'u(0.4,t)':'{:.4f}', 'D∞ к BE (r=0.4)':'{:.4e}',
}).set_caption('Профили при совпадающих физических временах'))

for r in rs:
    cn_times, cn_vals = heat_records['CN',r]
    print(f'КН, r={r:g}:  u(0.4,t_n) для n=0..4 → '
          + ', '.join(f'{v:.3f}' for v in cn_vals[:5,i_edge]))

# Без искусственного отсечения: регистрируем действительный диапазон результатов.
raw_min = min(float(v.min()) for _,v in heat_records.values())
raw_max = max(float(v.max()) for _,v in heat_records.values())
print(f'Диапазон на всех рассчитанных слоях: [{raw_min:.6f}, {raw_max:.6f}].')

### Анимация 2 — одинаковое время, разные шаги

Верхняя панель: неявный Эйлер; нижняя: Кранк—Николсон. В каждой три значения $r$ и начальный контур (тонкий пунктир). **Все шесть кривых обновляются в одинаковый момент** $t=0,0.002,\dots,0.1$ — нет подмены физического времени номером слоя.

In [ ]:
common_times = np.linspace(0, 0.1, 51)
fig, axs = plt.subplots(2, 1, figsize=(10, 7.2), sharex=True, sharey=True)
animated = {}
for ax, method in zip(axs, methods_heat):
    ax.plot(x_heat, heat0, color='0.7', lw=1, ls=':', label='Начальный профиль')
    for r in rs:
        line, = ax.plot([], [], lw=2.0, label=f'r={r:g}')
        animated[method,r] = line
    ax.set(ylabel='u(x,t)', xlim=(0.3,0.7), ylim=(-0.075,1.075),
           title='Эйлер назад' if method=='BE' else 'Кранк—Николсон')
    ax.legend(loc='upper right', ncol=4, fontsize=8)
axs[-1].set_xlabel('x')
clock = fig.text(0.5, 0.985, '', ha='center', va='top', fontsize=12)
fig.tight_layout(rect=[0,0,1,0.97])

def update_heat(frame):
    t = float(common_times[frame])
    artists = []
    for method in methods_heat:
        for r in rs:
            line = animated[method,r]
            line.set_data(x_heat, at_time(heat_records[method,r],t))
            artists.append(line)
    clock.set_text(f'Общее физическое время t = {t:.3f}')
    return (*artists, clock)

anim_heat = FuncAnimation(fig, update_heat, frames=len(common_times),
                          interval=100, blit=True, repeat=True)
display(HTML(anim_heat.to_jshtml(fps=10, default_mode='loop')))
plt.close(fig)

## 4. Выводы

1. ** Сходимость:** при $r=0.4$, $\Delta t=O(\Delta x^2)$; FTCS и неявный Эйлер демонстрируют $p\approx2$ на достаточно тонких сетках. Обе схемы устойчивы в заданном режиме.
2. **Исчезновение деталей:** пятая гармоника затухает в $25$ раз быстрее первой по показателю экспоненты: множители $e^{-25\pi^2t}$ и $e^{-\pi^2t}$. Поэтому мелкие детали расплываются быстрее крупных — это свойство **самого уравнения теплопроводности**, а не только численной схемы.
3. **Эйлер назад:** устойчив при любом $r>0$, сохраняет дискретный принцип максимума и не создаёт знакопеременных высокочастотных мод. Крупный шаг увеличивает временную погрешность на ранних временах.
4. **Кранк—Николсон:** безусловная $\ell_2$-устойчивость **не означает** отсутствие осцилляций или достаточную точность. При $r=10$ в узле $x=0.4$ первые значения примерно $1\to0.218\to0.791$: знак чередования изменений очевиден, хотя в данном эксперименте значения остаются неотрицательными. Отрицательные значения намеренно не исправляются.
5. **Причина:** для высокочастотной пространственной моды коэффициент перехода Кранка—Николсона близок к $(1-2r)/(1+2r)$. При $r=10$ это $-19/21\approx-0.905$: знак высокочастотной компоненты меняется каждый шаг и её затухание медленно. Для неявного Эйлера коэффициент $1/(1+4r)$ положителен и мал.

**Итог:** проверка только устойчивости недостаточна. Нужны совпадающие физические времена, оценка ошибки и контроль формы решения у разрывов.